# Exploratory Data Analysis — AQI & Weather Patterns

This notebook explores the raw AQI and weather data to identify trends,
seasonal patterns, and correlations that inform feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
print('Libraries loaded')

## 1. Load Data from Hopsworks Feature Store

In [ ]:
from src.hopsworks_setup import get_feature_store
from src.training_pipeline.loader import load_feature_view

fs = get_feature_store()
df, _ = load_feature_view(fs)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')
df.head()

## 2. AQI Distribution Analysis

In [ ]:
if 'aqi' in df.columns:
    fig = make_subplots(rows=1, cols=2, subplot_titles=['AQI Distribution', 'AQI by City'])
    fig.add_trace(go.Histogram(x=df['aqi'], nbinsx=50, name='AQI'), row=1, col=1)
    fig.add_trace(go.Box(x=df['city'], y=df['aqi'], name='AQI'), row=1, col=2)
    fig.update_layout(height=400, showlegend=False)
    fig.show()
else:
    print('AQI column not found in dataset')

## 3. Time Series Patterns

In [ ]:
if 'timestamp' in df.columns and 'aqi' in df.columns:
    df_ts = df.set_index('timestamp').sort_index()
    # Daily average
    daily = df_ts.groupby('city')['aqi'].resample('D').mean().reset_index()
    fig = px.line(daily, x='timestamp', y='aqi', color='city',
                  title='Daily Average AQI by City')
    fig.update_layout(height=400)
    fig.show()

## 4. Correlation Matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                title='Feature Correlation Matrix',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.update_layout(height=600)
fig.show()

## 5. Diurnal AQI Pattern (Hour of Day)

In [ ]:
if 'hour' in df.columns:
    hourly_avg = df.groupby(['city', 'hour'])['aqi'].mean().reset_index()
    fig = px.line(hourly_avg, x='hour', y='aqi', color='city',
                  title='Average AQI by Hour of Day',
                  markers=True)
    fig.update_layout(height=400, xaxis_dtick=1)
    fig.show()

## 6. Missing Data Assessment

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df[missing_df['Missing'] > 0]

## 7. Insights & Next Steps

- Note which features have strong correlations with AQI
- Identify seasonal patterns (monsoon vs winter AQI)
- Check for data gaps that need interpolation
- Determine optimal lag window for feature engineering